### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import copy

from sklearn.metrics import accuracy_score

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.cnn_regressor import CNNRegressor

### CONFIGURATION

In [5]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_image_data(
    active_dataset
)

y_test = np.asarray(test.dataset.targets)[test.indices]

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

y_train, y_valid = processor.transform_target(y_train, y_valid)

test = processor.convert_to_numpy(test) 

### GRADIENT BOOSTING

In [6]:
small_gb_preds = pd.read_csv(f"{MODELS_PATH}/{active_dataset}/2026_03_27_09_01/predictions.csv")
print("GB small CNN:", accuracy_score(y_test, small_gb_preds[active_dataset_config["target"]]))

big_gb_preds = pd.read_csv(f"{MODELS_PATH}/{active_dataset}/2026_04_07_20_09/predictions.csv")
print("GB big CNN:", accuracy_score(y_test, big_gb_preds[active_dataset_config["target"]]))

GB small CNN: 0.6894
GB big CNN: 0.7641


### CONVOLUTIONAL NEURAL NETWORKS

Big CNN total parameters: 259,914

Small CNN total parameters: 16,986

In [11]:
class CNN(CNNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(self, X_train, y_train, X_valid, y_valid, patience=5):
        
        in_channels = X_train.shape[1]
        output_size = int(np.max(y_train)) + 1

        image_size = X_train.shape[-1]

        conv1_out = image_size - self.kernel_size + 1
        pool1_out = conv1_out // self.pool_size

        conv2_out = pool1_out - self.kernel_size + 1
        pool2_out = conv2_out // self.pool_size

        linear_input = self.channels[1] * pool2_out * pool2_out
        
        self._get_network(in_channels, linear_input, output_size)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                pred_labels = val_preds.argmax(dim=1)
                val_acc = (pred_labels == y_valid_t).float().mean().item()

            if (epoch + 1) % 1 == 0:
                print(f"Epoch: {epoch + 1} | Validation Log Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

In [12]:
big_cnn = CNN(epochs=100, learning_rate=0.001, channels=[32, 64], kernel_size=5, pool_size=2, hidden_size=128, batch_size=256)
big_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

raw_preds = big_cnn.predict(test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch: 1 | Validation Log Loss: 1.4944 | Validation Accuracy: 0.4612
Epoch: 2 | Validation Log Loss: 1.4047 | Validation Accuracy: 0.4871
Epoch: 3 | Validation Log Loss: 1.2263 | Validation Accuracy: 0.5633
Epoch: 4 | Validation Log Loss: 1.2051 | Validation Accuracy: 0.5629
Epoch: 5 | Validation Log Loss: 1.1554 | Validation Accuracy: 0.5956
Epoch: 6 | Validation Log Loss: 1.0820 | Validation Accuracy: 0.6174
Epoch: 7 | Validation Log Loss: 1.0685 | Validation Accuracy: 0.6234
Epoch: 8 | Validation Log Loss: 1.0356 | Validation Accuracy: 0.6375
Epoch: 9 | Validation Log Loss: 1.0169 | Validation Accuracy: 0.6436
Epoch: 10 | Validation Log Loss: 1.0655 | Validation Accuracy: 0.6319
Epoch: 11 | Validation Log Loss: 1.0354 | Validation Accuracy: 0.6473
Epoch: 12 | Validation Log Loss: 0.9606 | Validation Accuracy: 0.6717
Epoch: 13 | Validation Log Loss: 0.9524 | Validation Accuracy: 0.6692
Epoch: 14 | Validation Log Loss: 0.9463 | Validation Accuracy: 0.6731
Epoch: 15 | Validation Log Lo

In [13]:
small_cnn = CNN(epochs=100, learning_rate=0.001, channels=[8, 16], kernel_size=5, pool_size=2, hidden_size=32, batch_size=256)
small_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

raw_preds = small_cnn.predict(test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch: 1 | Validation Log Loss: 1.7002 | Validation Accuracy: 0.3900
Epoch: 2 | Validation Log Loss: 1.5622 | Validation Accuracy: 0.4384
Epoch: 3 | Validation Log Loss: 1.5171 | Validation Accuracy: 0.4510
Epoch: 4 | Validation Log Loss: 1.4619 | Validation Accuracy: 0.4844
Epoch: 5 | Validation Log Loss: 1.4579 | Validation Accuracy: 0.4823
Epoch: 6 | Validation Log Loss: 1.4371 | Validation Accuracy: 0.4888
Epoch: 7 | Validation Log Loss: 1.4012 | Validation Accuracy: 0.5038
Epoch: 8 | Validation Log Loss: 1.3744 | Validation Accuracy: 0.5152
Epoch: 9 | Validation Log Loss: 1.3396 | Validation Accuracy: 0.5242
Epoch: 10 | Validation Log Loss: 1.3606 | Validation Accuracy: 0.5152
Epoch: 11 | Validation Log Loss: 1.3355 | Validation Accuracy: 0.5284
Epoch: 12 | Validation Log Loss: 1.3025 | Validation Accuracy: 0.5427
Epoch: 13 | Validation Log Loss: 1.2748 | Validation Accuracy: 0.5514
Epoch: 14 | Validation Log Loss: 1.2942 | Validation Accuracy: 0.5471
Epoch: 15 | Validation Log Lo